In [1]:
import os 
import numpy as np 
import torch

In [2]:
# Measure torch cuda event time
def measure_time(func, *args, **kwargs):
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    # Record the start event
    start_event.record()

    # Call the function
    result = func(*args, **kwargs)

    # Record the end event
    end_event.record()

    # Wait for the events to be recorded
    torch.cuda.synchronize()

    # Calculate elapsed time in milliseconds
    elapsed_time_ms = start_event.elapsed_time(end_event)

    return elapsed_time_ms, result

In [39]:
X = torch.randn(2048, 2048, device='cuda', dtype=torch.bfloat16)
S = torch.tensor(1.0, dtype=torch.bfloat16, device='cuda')  

# warm up
for _ in range(10):
    X_q = torch.clamp(torch.round(X / S), -128, 127).to(torch.int8)

n_iter = 1_000
start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)
start_event.record()
for _ in range(n_iter):
    X_q = torch.clamp(torch.round(X / S), -128, 127).to(torch.int8)
end_event.record()
torch.cuda.synchronize()
elapsed_time_ms = start_event.elapsed_time(end_event)
print(f"Avg quantization time over {n_iter} iterations: {elapsed_time_ms / n_iter} ms")

Avg quantization time over 1000 iterations: 0.03362815856933594 ms


In [40]:
X = torch.randn(2048, 2048, device='cuda', dtype=torch.bfloat16)
S = torch.randn((2048,), dtype=torch.bfloat16, device='cuda')  

# warm up
for _ in range(10):
    X_q = torch.clamp(torch.round(X / S[None, :]), -128, 127).to(torch.int8)

n_iter = 1_000
start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)
start_event.record()
for _ in range(n_iter):
    X_q = torch.clamp(torch.round(X / S[None, :]), -128, 127).to(torch.int8)
end_event.record()
torch.cuda.synchronize()
elapsed_time_ms = start_event.elapsed_time(end_event)
print(f"Avg quantization time over {n_iter} iterations: {elapsed_time_ms / n_iter} ms")

Avg quantization time over 1000 iterations: 0.03472480010986328 ms
